# Week 8 (Notebook 1): Decoder‑Only Generation (GPT)

This notebook has two goals:

1. **Architecture intuition (PyTorch):** implement a tiny decoder‑only Transformer (causal self‑attention) to understand the forward pass + masking + next‑token loss.
2. **Power of pretraining (Hugging Face):** fine‑tune a small pretrained GPT model on a small dataset slice with **CPU‑friendly defaults**.

Notes:
- First run will download a dataset/model from Hugging Face.
- CPU training is intentionally small (few steps, short sequence length) so it finishes in a reasonable time.

In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset

from utils import TinyGPT, evaluate_tiny_gpt, get_device, train_tiny_gpt, clean_memory

seed = 204
torch.manual_seed(seed)

device = get_device()
device

device(type='mps')

## Part A — Tiny decoder‑only Transformer in pure PyTorch

The tiny model in `utils.py` follows the same high-level decoder-only pattern as GPT:

1. **Token + position embeddings**
   - `input_ids` have shape `(batch, seq_len)`.
   - Token embeddings produce `(batch, seq_len, d_model)`.
   - Position embeddings are learned with shape `(max_seq_len, d_model)`, where `max_seq_len` is the model context window.
   - We add positions with `self.position_embedding(position).unsqueeze(0)` so the position tensor broadcasts across the batch.

2. **Causal self-attention**
   - Each token can attend only to itself and earlier tokens.
   - The causal mask prevents position `t` from seeing positions `> t`.
   - There is no padding mask in this Part A toy setup because every training window has the same fixed length.
   - **Note on Inference (KV Cache):** During fast autoregressive generation, models only pass one new token at a time (`query_len=1`) and retrieve past keys/values from memory (`key_len=N`). The causal mask calculates a diagonal offset (`key_len - query_len`) to allow the single query token to attend to the entire history.

3. **Pre-norm Transformer blocks with residual connections**
   - Each block uses the pattern `x = x + Attention(LayerNorm(x))`.
   - Then it uses `x = x + FeedForward(LayerNorm(x))`.
   - Those two `+ x` paths are the residual connections. By the time we leave the stack, `x` already contains the residual-updated hidden states.

4. **Final normalization + language-model head**
   - After all Transformer blocks, we apply `final_norm` once: `hidden_states = self.final_norm(x)`.
   - Then `lm_head` projects each hidden state to vocabulary logits: `logits = self.lm_head(hidden_states)`.
   - We do **not** add another residual connection around `final_norm`; the final norm is not an attention or feed-forward sublayer.
   - The LM head uses `bias=False`. We also apply **Weight Tying** (`self.lm_head.weight = self.embedding.weight`), where the input embedding and output projection share the exact same matrix. This mathematically acts as a reverse lookup and saves massive amounts of parameters.

5. **Next-token prediction loss**
   - The target is the same character window shifted one step to the right.
   - The model learns: given tokens up to position `t`, predict token `t + 1`.

In [2]:
# Tiny demo dataset (character-level) so everything is self-contained.
text = """
to be or not to be.
this is a tiny corpus for a tiny gpt.
we only want to demonstrate causal masking and next-token loss.
""".strip().lower()

chars = sorted(list(set(text)))
char_index_mapping = {ch: i for i, ch in enumerate(chars)}
index_char_mapping = {i: ch for ch, i in char_index_mapping.items()}

data = torch.tensor([char_index_mapping[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_length = 64

# Each x predicts the same window shifted one token to the right.
x_all = torch.stack([data[i : i + max_seq_length] for i in range(len(data) - max_seq_length)])
y_all = torch.stack([data[i + 1 : i + max_seq_length + 1] for i in range(len(data) - max_seq_length)])
train_loader = DataLoader(TensorDataset(x_all, y_all), batch_size=16, shuffle=True)

vocab_size, len(data), len(train_loader)

(26, 121, 4)

In [3]:
# CPU-friendly tiny training loop (few epochs over a tiny next-token dataset)
clean_memory()
model = TinyGPT(
    vocab_size=vocab_size,
    d_model=128,
    num_heads=4,
    mlp_hidden_dim=4 * 128,
    num_layers=2,
    pad_id=None,
    dropout=0.1,
    max_seq_len=max_seq_length,
)
model

Memory cleaned.


TinyGPT(
  (embedding): Embedding(26, 128)
  (position_embedding): Embedding(64, 128)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (self_attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (self_attn): MultiHeadAttention(
        (q_proj): Linear(in_features=128, out_features=128, bias=True)
        (k_proj): Linear(in_features=128, out_features=128, bias=True)
        (v_proj): Linear(in_features=128, out_features=128, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (ffn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=512, out_features=128, bias=True)
          (4): Dropout(p=0.1, inplace=False)
        )
  

### Decoding with temperature

`evaluate_tiny_gpt` generates text one token at a time. At each step it keeps only the model's last-position logits, rescales them by `temperature`, turns them into probabilities with softmax, and samples the next token with `torch.multinomial`.

If the vocabulary logits for the next token are $z_1, z_2, \ldots, z_V$ and the temperature is $T$, the sampled distribution is:

$$p_i = \frac{\exp(z_i / T)}{\sum_{j=1}^{V} \exp(z_j / T)}$$

- `temperature=1.0`: use the model's probabilities as-is.
- `temperature<1.0`: sharpen the distribution, making high-probability tokens more likely.
- `temperature>1.0`: flatten the distribution, making generation more random.
- `temperature -> 0`: Equivalent to greedy (argmax)

The sampling line is:

$$x_{t+1}^{(s)} \sim \operatorname{Categorical}(p), \quad s = 1, \ldots, \text{num\_samples}$$

In this notebook, `num_samples=1`, so the decoder chooses one next character per generation step, appends it to the context, and repeats until `max_new_tokens` characters have been generated.


In [5]:
model = train_tiny_gpt(
    model=model,
    device=device,
    train_loader=train_loader,
    epochs=250,
    lr=3e-4,
)

prompt = "to be"
idx0 = torch.tensor([[char_index_mapping[c] for c in prompt.lower()]], dtype=torch.long).to(device)
out = evaluate_tiny_gpt(model, idx0, max_new_tokens=100, max_seq_len=max_seq_length, temperature=1.0)[0].tolist()
print("".join(index_char_mapping[i] for i in out))

epoch 1 | loss 0.1639
epoch 11 | loss 0.0429
epoch 21 | loss 0.0349
epoch 31 | loss 0.0267
epoch 41 | loss 0.0282
epoch 51 | loss 0.0233
epoch 61 | loss 0.0362
epoch 71 | loss 0.0250
epoch 81 | loss 0.0345
epoch 91 | loss 0.0325
epoch 101 | loss 0.0183
epoch 111 | loss 0.0310
epoch 121 | loss 0.0317
epoch 131 | loss 0.0129
epoch 141 | loss 0.0252
epoch 151 | loss 0.0281
epoch 161 | loss 0.0268
epoch 171 | loss 0.0282
epoch 181 | loss 0.0322
epoch 191 | loss 0.0242
epoch 201 | loss 0.0279
epoch 211 | loss 0.0285
epoch 221 | loss 0.0304
epoch 231 | loss 0.0306
epoch 241 | loss 0.0319
to be.
this is a tiny corpus for a tiny gpt.
we only want to demonstrate causal masking and next-token lo


## Part B — Fine‑tuning a pretrained GPT model (Hugging Face)

We now fine‑tune a pretrained decoder‑only model to show how much better it is with transfer learning.

**CPU‑realistic defaults**
- use `distilgpt2`
- short `block_size`
- small dataset slice
- `max_steps` instead of full epochs
- small batch size + gradient accumulation

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

hf_model_name = "distilgpt2"
block_size = 128

# Keep this small for CPU
n_train = 4000
n_val = 500

raw = load_dataset("wikitext", "wikitext-2-v1")
raw

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], return_attention_mask=False)

tok = raw.map(tokenize_fn, batched=True, remove_columns=raw["train"].column_names)

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_ds = tok.map(group_texts, batched=True)

# Shuffle then take small CPU-friendly slices
train_ds = lm_ds["train"].shuffle(seed=seed).select(range(min(n_train, len(lm_ds["train"])) ))
val_ds = lm_ds["validation"].shuffle(seed=seed).select(range(min(n_val, len(lm_ds["validation"])) ))
train_ds, val_ds

In [ ]:
model = AutoModelForCausalLM.from_pretrained(hf_model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

out_dir = "./models/week8_distilgpt2_wikitext2"
os.makedirs(out_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=out_dir,
    overwrite_output_dir=True,
    max_steps=300,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=seed,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

trainer

In [ ]:
# CPU-realistic run: keep this small
# trainer.train()

# If you already trained, point to a checkpoint folder here:
# ckpt = "./models/week8_distilgpt2_wikitext2/checkpoint-300"
# model = AutoModelForCausalLM.from_pretrained(ckpt).to(device)

print("Ready: uncomment trainer.train() to fine-tune.")

In [ ]:
# Generation demo (works before/after fine-tuning)
from transformers import set_seed

set_seed(seed)
prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

gen = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    top_p=0.95,
    temperature=0.9,
)
print(tokenizer.decode(gen[0], skip_special_tokens=True))